In [ ]:
'''Analyze and visualize feature importance in tree-based models (Random Forest, XGBoost, LightGBM) using impurity-based metrics, 
permutation importance, and SHAP values to guide feature engineering and selection.'''

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import shap

In [ ]:
# load dataset and prepare sample features
df = pd.read_csv('global_pharmacy_sales_data.csv')
df_sample = df.sample(n=5000, random_state=42).copy()
for col in ['category', 'age_group', 'region']:
    df_sample[col] = LabelEncoder().fit_transform(df_sample[col])

features = ['unit_price', 'stock_level', 'expiry_days_remaining', 'covid_flag', 'category', 'age_group', 'region', 'month']
X = df_sample[features]
y = df_sample['units_sold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# train tree based models
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)

xgb = XGBRegressor(n_estimators=50, random_state=42)
xgb.fit(X_train, y_train)

lgb = LGBMRegressor(n_estimators=50, random_state=42, verbose=-1)
lgb.fit(X_train, y_train)

In [ ]:
# visualize impurity based feature importance across models
imp_df = pd.DataFrame({
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb.feature_importances_,
    'LightGBM': lgb.feature_importances_ / lgb.feature_importances_.sum()
}, index=features)

plt.figure(figsize=(9, 4))
imp_df.plot(kind='bar', figsize=(9, 4))
plt.title('impurity based feature importance')
plt.tight_layout()
plt.show()

In [ ]:
# calculate and plot permutation importance
perm_imp = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=1)
perm_series = pd.Series(perm_imp.importances_mean, index=features).sort_values()

plt.figure(figsize=(8, 4))
plt.barh(perm_series.index, perm_series.values, color='teal')
plt.title('permutation feature importance')
plt.tight_layout()
plt.show()

In [ ]:
# calculate and plot shap values for tree model
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test.iloc[:100])

plt.figure(figsize=(8, 4))
shap.summary_plot(shap_values, X_test.iloc[:100], plot_type='bar', show=False)
plt.title('shap feature importance')
plt.tight_layout()
plt.show()

In [ ]:
# compare feature importance metrics to guide feature selection
shap_imp = np.abs(shap_values).mean(axis=0)
summary_df = pd.DataFrame({
    'rf_impurity': rf.feature_importances_,
    'xgb_impurity': xgb.feature_importances_,
    'permutation': perm_imp.importances_mean,
    'shap': shap_imp
}, index=features)

print(summary_df.rank(ascending=False))
selected_features = summary_df.mean(axis=1).sort_values(ascending=False).head(5).index.tolist()
print('selected top features:', selected_features)